# Dataset and experimental design: manifest-level figures

Figures produced:

1. **Domain metadata overview** -- distribution of `road_type` and `time_of_day`
   across the full ZOD train split.
2. **Class prevalence by road type** -- Pedestrian / VulnerableVehicle / Vehicle
   presence rates across the five road types.
3. **Stream structure visualization** -- stripe plot showing domain blocks in the
   forward and reverse orderings, with the bootstrap prefix highlighted.
4. **Bootstrap composition comparison** -- city-day vs city-mixed bootstrap sets.
5. **Federated client partition** -- domain composition per client under contiguous
   partitioning.

All figures are saved as PDF to `notes/figures/`.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 140,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.facecolor": "white",
    "text.usetex": False,
    "mathtext.fontset": "cm",
})

sys.path.insert(0, str(Path.cwd() / "notebooks"))
if not (Path.cwd() / "pyproject.toml").exists():
    sys.path.insert(0, str(Path.cwd().parent / "notebooks"))

import analysis_helpers as ah

PROJECT_ROOT = ah.find_project_root()
FIG_DIR = PROJECT_ROOT / "notes" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = ah.resolve_manifest_path(
    PROJECT_ROOT, "data/ZOD_frames_preprocessed/Frames_1600x480/manifest_cityday_road_type.json"
).parent

print("Manifest dir:", MANIFEST_DIR)
print("Figure dir:  ", FIG_DIR)

In [ ]:
def load(name: str) -> Dict[str, Any]:
    with open(MANIFEST_DIR / name) as f:
        return json.load(f)

MANIFESTS = {
    "cityday_road_type": load("manifest_cityday_road_type.json"),
    "cityday_reverse": load("manifest_cityday_reverse.json"),
    "citymix_road_type": load("manifest_citymix_road_type.json"),
    "citymix_reverse": load("manifest_citymix_reverse.json"),
    "citymix_conditions": load("manifest_citymix_conditions.json"),
}

REF = MANIFESTS["cityday_road_type"]
ALL_FRAMES = REF["frames"]
BOOT_N = REF["ordering"]["bootstrap_frames"]

ROAD_TYPE_ORDER = ["city", "arterial-urban", "highway", "arterial-rural", "smaller-rural"]
TOD_ORDER = ["day", "twilight", "night"]
TARGET_CLASSES = ["Vehicle", "Pedestrian", "VulnerableVehicle"]

DOMAIN_COLORS = {
    "city": "#1f77b4",
    "arterial-urban": "#ff7f0e",
    "highway": "#2ca02c",
    "arterial-rural": "#d62728",
    "smaller-rural": "#9467bd",
    "clear": "#FDDA0D",
    "cloudy": "#A9A9A9",
    "fog": "#C4C3D0",
    "rain_wet": "#4682B4",
    "snow": "#E0F0FF",
}

TOD_COLORS = {
    "day": "#FFD700",
    "twilight": "#FF8C00",
    "night": "#191970",
}

ROAD_SHORT = {
    "city": "City",
    "arterial-urban": "Art.\u2013Urban",
    "highway": "Highway",
    "arterial-rural": "Art.\u2013Rural",
    "smaller-rural": "Sm.\u2013Rural",
}

COND_SHORT = {
    "clear": "Clear",
    "cloudy": "Cloudy",
    "fog": "Fog",
    "rain_wet": "Rain/Wet",
    "snow": "Snow",
}


def _get_bootstrap_train_frames(mkey: str) -> List[Dict[str, Any]]:
    """Return the first N *train* frames from a manifest (skipping val)."""
    m = MANIFESTS[mkey]
    n = m["ordering"]["bootstrap_frames"]
    train_frames_out: List[Dict[str, Any]] = []
    for f in m["frames"]:
        if f.get("split") == "train":
            train_frames_out.append(f)
            if len(train_frames_out) >= n:
                break
    return train_frames_out


print(f"Total frames: {len(ALL_FRAMES)}, bootstrap: {BOOT_N}")

## 1  Domain metadata overview

Distribution of `road_type` and `time_of_day` across the full ZOD train split
(all frames in the manifest, including bootstrap).

In [ ]:
train_frames = [f for f in ALL_FRAMES if f.get("split") == "train"]
n_train = len(train_frames)

rt_counts = Counter(f["road_type"] for f in train_frames)
tod_counts = Counter(f["time_of_day"] for f in train_frames)

fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))

# Road type
ax = axes[0]
labels = [ROAD_SHORT.get(r, r) for r in ROAD_TYPE_ORDER]
vals = [rt_counts.get(r, 0) for r in ROAD_TYPE_ORDER]
colors = [DOMAIN_COLORS[r] for r in ROAD_TYPE_ORDER]
bars = ax.barh(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
            f"${v:,}$ ({v / n_train:.0%})", va="center", fontsize=7)
ax.set_xlabel("Number of frames")
ax.set_title(r"$\mathbf{Road\ type}$ distribution")
ax.invert_yaxis()
ax.set_xlim(0, max(vals) * 1.28)

# Time of day
ax = axes[1]
labels_tod = [t.capitalize() for t in TOD_ORDER]
vals_tod = [tod_counts.get(t, 0) for t in TOD_ORDER]
colors_tod = [TOD_COLORS[t] for t in TOD_ORDER]
bars = ax.barh(labels_tod, vals_tod, color=colors_tod, edgecolor="white", linewidth=0.5)
for bar, v in zip(bars, vals_tod):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
            f"${v:,}$ ({v / n_train:.0%})", va="center", fontsize=7)
ax.set_xlabel("Number of frames")
ax.set_title(r"$\mathbf{Time\ of\ day}$ distribution")
ax.invert_yaxis()
ax.set_xlim(0, max(vals_tod) * 1.28)

fig.suptitle(f"ZOD train split metadata ($n = {n_train:,}$ frames)", fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "domain_metadata_overview.pdf", bbox_inches="tight")
plt.show()

## 2  Target-class prevalence by road type

Fraction of frames containing each target class, broken down by `road_type`.
This motivates the domain-shift hypothesis: pedestrians are concentrated in
urban environments, while highway and rural areas see dramatically lower
pedestrian prevalence.

In [ ]:
rt_class_rates: Dict[str, Dict[str, float]] = {}
for rt in ROAD_TYPE_ORDER:
    rt_frames = [f for f in train_frames if f["road_type"] == rt]
    n = len(rt_frames)
    rt_class_rates[rt] = {}
    for cls in TARGET_CLASSES:
        count = sum(1 for f in rt_frames if cls in f.get("categories_present", []))
        rt_class_rates[rt][cls] = count / n if n > 0 else 0.0

CLASS_LABELS = {"Vehicle": "Vehicle", "Pedestrian": "Pedestrian", "VulnerableVehicle": "Vuln. Vehicle"}

fig, ax = plt.subplots(figsize=(7, 3.2))
x = np.arange(len(ROAD_TYPE_ORDER))
width = 0.25
class_colors = {"Vehicle": "#1f77b4", "Pedestrian": "#d62728", "VulnerableVehicle": "#ff7f0e"}

for i, cls in enumerate(TARGET_CLASSES):
    vals = [rt_class_rates[rt][cls] for rt in ROAD_TYPE_ORDER]
    bars = ax.bar(x + i * width, vals, width, label=CLASS_LABELS[cls],
                  color=class_colors[cls], edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        if v > 0.05:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{v:.0%}", ha="center", va="bottom", fontsize=6)

ax.set_xticks(x + width)
ax.set_xticklabels([ROAD_SHORT.get(r, r) for r in ROAD_TYPE_ORDER])
ax.set_ylabel("Fraction of frames containing class")
ax.set_title("Target-class prevalence by road type")
ax.set_ylim(0, 1.12)
ax.legend(loc="upper center", ncol=3, framealpha=0.9, bbox_to_anchor=(0.5, 1.0))
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "class_prevalence_by_road_type.pdf", bbox_inches="tight")
plt.show()

## 3  Stream structure visualization

Stripe plots showing the domain blocks in the ordered stream, with the
bootstrap prefix highlighted.  Each row is a different manifest variant,
giving the reader an immediate visual sense of block structure and ordering.

In [ ]:
def _block_label(bname: str) -> str:
    return (ROAD_SHORT.get(bname) or COND_SHORT.get(bname) or bname)


def stripe_plot(
    manifest_specs: List[Tuple[str, str, str]],
    title: str,
    figsize: Tuple[float, float] = (9, 2.4),
    save_name: str | None = None,
) -> None:
    """Draw a horizontal stripe for each manifest showing bootstrap + domain blocks."""
    fig, ax = plt.subplots(figsize=figsize)
    row_height = 0.6
    gap = 0.15

    used_labels: set = set()
    legend_handles: list = []

    for row_i, (label, mkey, _color_field) in enumerate(manifest_specs):
        m = MANIFESTS[mkey]
        o = m["ordering"]
        boot = o["bootstrap_frames"]
        block_order = o["block_order"]
        block_sizes = o["block_sizes"]
        total = boot + sum(block_sizes.values())

        y_bottom = row_i * (row_height + gap)

        ax.barh(y_bottom, boot, height=row_height, left=0,
                color="#cccccc", edgecolor="white", linewidth=0.3)
        if "Bootstrap" not in used_labels:
            legend_handles.append(mpatches.Patch(color="#cccccc", label="Bootstrap"))
            used_labels.add("Bootstrap")

        pos = boot
        for bname in block_order:
            sz = block_sizes[bname]
            c = DOMAIN_COLORS.get(bname, "#888888")
            ax.barh(y_bottom, sz, height=row_height, left=pos,
                    color=c, edgecolor="white", linewidth=0.3)
            if sz / total > 0.04:
                ax.text(pos + sz / 2, y_bottom + row_height / 2,
                        _block_label(bname),
                        ha="center", va="center", fontsize=5.5, color="black")
            disp_name = _block_label(bname)
            if disp_name not in used_labels:
                legend_handles.append(mpatches.Patch(color=c, label=disp_name))
                used_labels.add(disp_name)
            pos += sz

    y_positions = [i * (row_height + gap) + row_height / 2 for i in range(len(manifest_specs))]
    y_labels = [spec[0] for spec in manifest_specs]
    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Frame index")
    ax.set_title(title)
    ax.legend(handles=legend_handles, fontsize=6.5, framealpha=0.9,
              ncol=min(len(legend_handles), 6),
              loc="lower center", bbox_to_anchor=(0.5, -0.38))

    fig.tight_layout()
    if save_name:
        fig.savefig(FIG_DIR / save_name, bbox_inches="tight")
    plt.show()


stripe_plot(
    [
        ("City-day, forward", "cityday_road_type", "road_type"),
        ("City-day, reverse", "cityday_reverse", "road_type"),
        ("City-mix, forward", "citymix_road_type", "road_type"),
        ("City-mix, conditions", "citymix_conditions", "conditions"),
    ],
    title="Stream block structure by manifest variant",
    figsize=(9, 3.2),
    save_name="stream_structure.pdf",
)

## 4  Bootstrap composition comparison

Side-by-side comparison of the city-day and city-mixed bootstrap sets.
The city-day bootstrap is nearly homogeneous (city roads, daytime), while
the city-mixed bootstrap introduces night and twilight frames.

In [ ]:
boot_cd = _get_bootstrap_train_frames("cityday_road_type")
boot_cm = _get_bootstrap_train_frames("citymix_road_type")

fig, axes = plt.subplots(2, 2, figsize=(8, 4.5))

for col, (bframes, blabel) in enumerate([(boot_cd, "City-day"), (boot_cm, "City-mixed")]):
    n = len(bframes)

    # Road type
    ax = axes[0, col]
    rt_c = Counter(f["road_type"] for f in bframes)
    labels = [ROAD_SHORT.get(r, r) for r in ROAD_TYPE_ORDER]
    vals = [rt_c.get(r, 0) for r in ROAD_TYPE_ORDER]
    colors = [DOMAIN_COLORS[r] for r in ROAD_TYPE_ORDER]
    bars = ax.barh(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
                    f"${v:,}$ ({v / n:.1%})", va="center", fontsize=6.5)
    ax.set_title(f"{blabel} bootstrap \u2014 road type")
    ax.invert_yaxis()
    ax.set_xlim(0, n * 1.2)

    # Time of day
    ax = axes[1, col]
    tod_c = Counter(f["time_of_day"] for f in bframes)
    labels_tod = [t.capitalize() for t in TOD_ORDER]
    vals_tod = [tod_c.get(t, 0) for t in TOD_ORDER]
    colors_tod = [TOD_COLORS[t] for t in TOD_ORDER]
    bars = ax.barh(labels_tod, vals_tod, color=colors_tod, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals_tod):
        if v > 0:
            ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
                    f"${v:,}$ ({v / n:.1%})", va="center", fontsize=6.5)
    ax.set_title(f"{blabel} bootstrap \u2014 time of day")
    ax.invert_yaxis()
    ax.set_xlim(0, n * 1.2)

fig.suptitle(f"Bootstrap composition comparison ($n = {BOOT_N:,}$ frames each)",
             fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "bootstrap_composition.pdf", bbox_inches="tight")
plt.show()

## 5  Federated client partition

Under contiguous partitioning with 4 clients, the stream (excluding the shared
bootstrap prefix) is split into 4 equal consecutive segments.  This creates
natural domain heterogeneity: some clients see mostly familiar city data, while
others receive highway and rural frames.

In [ ]:
NUM_CLIENTS = 4

o = REF["ordering"]
block_order = o["block_order"]
block_sizes = o["block_sizes"]
total_stream = sum(block_sizes.values())
per_client = total_stream // NUM_CLIENTS
remainder = total_stream % NUM_CLIENTS

block_bounds: List[Tuple[str, int, int]] = []
pos = 0
for b in block_order:
    sz = block_sizes[b]
    block_bounds.append((b, pos, pos + sz))
    pos += sz

client_ranges = []
for c in range(NUM_CLIENTS):
    s = c * per_client + min(c, remainder)
    e = s + per_client + (1 if c < remainder else 0)
    client_ranges.append((s, e))

fig, ax = plt.subplots(figsize=(9, 2.8))
row_height = 0.6
gap = 0.15

used_labels: set = set()
legend_handles: list = []

for ci, (cs, ce) in enumerate(client_ranges):
    y_bottom = ci * (row_height + gap)
    for bname, bstart, bend in block_bounds:
        overlap_start = max(cs, bstart)
        overlap_end = min(ce, bend)
        if overlap_end > overlap_start:
            w = overlap_end - overlap_start
            left = overlap_start - cs
            c = DOMAIN_COLORS.get(bname, "#888888")
            ax.barh(y_bottom, w, height=row_height, left=left,
                    color=c, edgecolor="white", linewidth=0.3)
            if w / per_client > 0.06:
                ax.text(left + w / 2, y_bottom + row_height / 2,
                        f"{_block_label(bname)}\n({w:,})",
                        ha="center", va="center", fontsize=5.5, color="black")
            disp = _block_label(bname)
            if disp not in used_labels:
                legend_handles.append(mpatches.Patch(color=c, label=disp))
                used_labels.add(disp)

y_positions = [i * (row_height + gap) + row_height / 2 for i in range(NUM_CLIENTS)]
y_labels = [f"Client {i}" for i in range(NUM_CLIENTS)]
ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels)
ax.set_xlabel("Frame index (within client)")
ax.set_title(f"Federated client partitioning ($K={NUM_CLIENTS}$, contiguous, forward ordering)")
ax.legend(handles=legend_handles, fontsize=6.5, framealpha=0.9,
          ncol=len(legend_handles),
          loc="lower center", bbox_to_anchor=(0.5, -0.32))

fig.tight_layout()
fig.savefig(FIG_DIR / "federated_client_partitions.pdf", bbox_inches="tight")
plt.show()

# Summary table
print(f"{'Client':<10} {'Frames':<8} {'Domains'}")
print("-" * 60)
for ci, (cs, ce) in enumerate(client_ranges):
    domains = {}
    for bname, bstart, bend in block_bounds:
        overlap = min(ce, bend) - max(cs, bstart)
        if overlap > 0:
            domains[bname] = overlap
    dom_str = ", ".join(f"{k}: {v:,}" for k, v in domains.items())
    print(f"Client {ci:<4} {ce - cs:<8,} {dom_str}")